# Cleveland Heart Disease Analysis & Prediction Pipeline
### Multi-Disease Prediction System using Machine Learning (Minor Project Part-I)
**Dataset:** Cleveland Heart Disease (1,025 records, 13 clinical features, binary diagnosis target)

---
### 1. Mathematical Formulation
* **Logistic Function:**
  $$P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \sum_{j=1}^{p} \beta_j X_j)}}$$
* **Log-Odds (Logit):**
  $$\ln\left(\frac{P}{1-P}\right) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p$$
* **Odds Ratio (OR):**
  $$OR_j = e^{\beta_j}$$
  * $OR_j > 1$: Higher values elevate risk of heart disease.
  * $OR_j < 1$: Higher values show an inverse/protective risk correlation.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.autolayout'] = True
%matplotlib inline

### 2. Data Loading & Inspection

In [ ]:
data_path = os.path.join('Dataset minor', 'heart.csv')
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())
df.info()

In [ ]:
print("Missing Values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("\nTarget Class Counts:")
print(df['target'].value_counts())
df.describe().T[['mean', 'std', 'min', '50%', 'max']]

### 3. Exploratory Data Analysis & Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['target'].value_counts()
labels = ['Heart Disease (1)', 'Healthy (0)']
axes[0].bar(labels, [counts[1], counts[0]], color=['#d9534f', '#2b5c8f'], width=0.5, edgecolor='black')
axes[0].set_title("Heart Disease Class Distribution", fontweight='bold')
axes[0].set_ylabel("Patient Count")

axes[1].pie([counts[1], counts[0]], labels=labels, autopct='%1.1f%%', colors=['#d9534f', '#2b5c8f'],
            explode=(0.04, 0), wedgeprops=dict(width=0.45, edgecolor='black'))
axes[1].set_title("Class Ratio Breakdown", fontweight='bold')
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="vlag", vmin=-0.5, vmax=0.5, square=True, linewidths=0.8)
plt.title("Pearson Correlation Heatmap - Cleveland Heart Disease", fontweight='bold', pad=12)
plt.show()

### 4. Train/Test Split (80/20 Stratified) & Standardization

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=23, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training Set: {X_train.shape[0]} samples | Testing Set: {X_test.shape[0]} samples")

### 5. Model Training, Odds Ratio Analysis & Benchmarking

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=23),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=23),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=23),
    'Support Vector Machine (RBF)': SVC(kernel='rbf', C=1.0, probability=True, random_state=23),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=23)
}

results = {}
for name, model in models.items():
    if name in ['Logistic Regression', 'Support Vector Machine (RBF)', 'K-Nearest Neighbors']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall (Sensitivity)': recall_score(y_test, y_pred),
        'Specificity': tn / (tn + fp),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

bench_df = pd.DataFrame(results).T
display(bench_df.style.highlight_max(axis=0, color='#d4efdf'))

### 6. Clinical Odds Ratio ($e^\beta$) Interpretation (Slide 1 Alignment)

In [ ]:
lr = models['Logistic Regression']
or_df = pd.DataFrame({
    'Feature': X.columns,
    'Beta': lr.coef_[0],
    'Odds Ratio (exp(Beta))': np.exp(lr.coef_[0])
}).sort_values(by='Odds Ratio (exp(Beta))', ascending=False)

plt.figure(figsize=(10, 5))
colors = ['#d9534f' if o >= 1.0 else '#2b5c8f' for o in or_df['Odds Ratio (exp(Beta))']]
plt.barh(or_df['Feature'], or_df['Odds Ratio (exp(Beta))'], color=colors, edgecolor='black')
plt.axvline(1.0, color='gray', linestyle='--', label='Baseline Risk (OR = 1.0)')
plt.title("Logistic Regression Odds Ratios (exp(Beta))", fontweight='bold')
plt.xlabel("Odds Ratio")
plt.legend()
plt.gca().invert_yaxis()
plt.show()

display(or_df)